[![Dataflowr](https://raw.githubusercontent.com/dataflowr/website/master/_assets/dataflowr_logo.png)](https://dataflowr.github.io/website/)

开始这份作业之前，请确保你熟悉 [模块 2a - PyTorch 张量](https://dataflowr.github.io/website/modules/2a-pytorch-tensors/) 和 [模块 2b - 自动微分](https://dataflowr.github.io/website/modules/2b-automatic-differentiation/) 的内容。


# 作业 1：从零实现 MLP

在这份作业里，你将用一个隐藏层的[多层感知机](https://en.wikipedia.org/wiki/Multilayer_perceptron)对 2D 空间中的点云进行分类。

请务必填写所有标注 `YOUR CODE HERE` 的地方。

建议：
- 尽量使用矩阵和向量运算（写出高效代码的好习惯）
- 如果你不熟悉 numpy，可以查一下 `np.max`、`np.clip`、`np.random.randn`、`np.reshape` 的文档。顺便说一下，矩阵乘法运算符是 `@`，你可能还想了解[广播规则](https://numpy.org/doc/stable/user/basics.broadcasting.html)，看看它是怎么处理不同尺寸张量之间的运算的
- 也可以查查 `torch.clamp`、`torch.nn.Parameter`

## 1. 一些工具函数和你的数据集

你不应该修改这一节的代码


In [ ]:
# 这些库都是用于绘图的
import numpy as np
import matplotlib.pyplot as plt

# 绘制数据集
def plot_data(ax, X, Y):
    plt.axis('off')
    ax.scatter(X[:, 0], X[:, 1], s=1, c=Y, cmap='bone')

from sklearn.datasets import make_moons
X, Y = make_moons(n_samples=2000, noise=0.1)

%matplotlib inline
x_min, x_max = -1.5, 2.5
y_min, y_max = -1, 1.5
fig, ax = plt.subplots(1, 1, facecolor='#4B6EA9')
ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)
plot_data(ax, X, Y)
plt.show()

这就是你的数据集：两个月亮，每个对应一个类别（上图中的黑色或白色）。

为了让实验更有趣、更直观，下面的代码可以让你看到分类器的决策边界。可惜的是，动画在 colab 上跑不了……


In [ ]:
# 定义评估分类器所用的网格
xx, yy = np.meshgrid(np.arange(x_min, x_max, .1),
                     np.arange(y_min, y_max, .1))

to_forward = np.array(list(zip(xx.ravel(), yy.ravel())))

# 绘制分类器的决策边界
def plot_decision_boundary(ax, X, Y, classifier):
    # 在网格上做前向传播，然后转成 numpy 以便绘图
    Z = classifier.forward(to_forward)
    Z = Z.reshape(xx.shape)
    
    # 绘制分类器在网格上取值的等高线
    ax.contourf(xx, yy, Z>0.5, cmap='Blues')
    
    # 然后绘制数据集
    plot_data(ax, X,Y)

## 2. 用 numpy 实现 MLP

这里你需要自己实现 [ReLU](https://en.wikipedia.org/wiki/Rectifier_(neural_networks) 激活函数和 [Sigmoid](https://en.wikipedia.org/wiki/Sigmoid_function)。


In [ ]:
class MyReLU(object):
    def forward(self, x):
        # relu 是 y_i = max(0, x_i)
        # 在这里写你的代码
        raise NotImplementedError()
        
    
    def backward(self, grad_output):
        # 对输入大于 0 的部分梯度为 1，其余为 0
        # 在这里写你的代码
        raise NotImplementedError()
    
    def step(self, learning_rate):
        # 这里什么都不用做，因为 ReLU 没有参数
        # 在这里写你的代码
        raise NotImplementedError()

class MySigmoid(object):
    def forward(self, x):
        # sigmoid 是 y_i = 1./(1+exp(-x_i))
        # 在这里写你的代码
        raise NotImplementedError()
    
    def backward(self, grad_output):
        # 偏导数是 e^-x / (e^-x + 1)^2
        # 在这里写你的代码
        raise NotImplementedError()
    
    def step(self, learning_rate):
        # 这里什么都不用做，因为 Sigmoid 没有参数
        # 在这里写你的代码
        raise NotImplementedError()

现在是个好时机，测试一下你的函数……


In [ ]:
test_relu = MyReLU()
test_relu.forward(X[10])

In [ ]:
test_relu.backward(np.ones(1))

In [ ]:
test_sig = MySigmoid()

test_sig.forward(np.ones(1))

In [ ]:
test_sig.backward(np.ones(1))

接下来要复杂一点：你需要实现自己的线性层，也就是乘以矩阵 W 再加上偏置 b。


In [ ]:
class MyLinear(object):
    def __init__(self, n_input, n_output):
        # 为 W 和 b 初始化两个随机矩阵（用 np.random.randn）
        # 在这里写你的代码
        raise NotImplementedError()

    def forward(self, x):
        # 保存一份 x 的副本，反向传播时会用到
        # 返回 xW + b
        # 在这里写你的代码
        raise NotImplementedError()

    def backward(self, grad_output):
        # y_i = \sum_j x_j W_{j,i}  + b_i
        # d y_i / d W_{j, i} = x_j
        # d loss / d y_i = grad_output[i]
        # 所以 d loss / d W_{j,i} = x_j * grad_output[i]  （链式法则）
        # 在这里写你的代码
        raise NotImplementedError()
        
        # d y_i / d b_i = 1
        # d loss / d y_i = grad_output[i]
        # 在这里写你的代码
        raise NotImplementedError()
        
        # 现在要计算对 x 的梯度，以便继续反向传播
        # d y_i / d x_j = W_{j, i}
        # 要计算损失的梯度，链式法则里要对所有可能的 y_i 求和
        # d loss / d x_j = \sum_i (d loss / d y_i) (d y_i / d x_j)
        # 在这里写你的代码
        raise NotImplementedError()
    
    def step(self, learning_rate):
        # 沿着保存的梯度的反方向更新 self.W 和 self.b，步长为 learning_rate
        # 在这里写你的代码
        raise NotImplementedError()

和实操里一样，现在你需要实现自己的网络（就是实操中我们叫 my_composition 的那个东西，见[实操](https://github.com/dataflowr/notebooks/blob/master/Module2/02_backprop.ipynb)）。注意，用了 Sigmoid 层，就应该用 BCE 损失。


In [ ]:
class Sequential(object):
    def __init__(self, layers):
        # 在这里写你的代码
        raise NotImplementedError()
        
    def forward(self, x):
        # 在这里写你的代码
        raise NotImplementedError()
    
    def compute_loss(self, out, label):
        # 使用 BCE 损失
        # -(label * log(output) + (1-label) * log(1-output))
        # 保存梯度并返回损失
        # 注意梯度里除以零的问题。
        # 把计算分成两种情况：标签为 0 和标签为 1
        # 给分母加上一个很小的值（1e-10）
        # 在这里写你的代码
        raise NotImplementedError()

    def backward(self):
        # 从损失的梯度开始，按顺序反向传播
        # 在这里写你的代码
        raise NotImplementedError()
    
    def step(self, learning_rate):
        # 对每一层做一个梯度步
        # 在这里写你的代码
        raise NotImplementedError()

In [ ]:
h=50

# 用你的 Sequential 定义网络
# 应该是一个 2 输入 h 输出的线性层，接一个 ReLU
# 然后是一个 h 输入 1 输出的线性层，接一个 sigmoid
# 欢迎尝试其他架构

# 在这里写你的代码
raise NotImplementedError()

In [ ]:
# 遗憾的是动画在 colab 上跑不了
# 如果在 colab 上，你应该注释掉下面这一行
%matplotlib notebook
fig, ax = plt.subplots(1, 1, facecolor='#4B6EA9')
ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)
losses = []
learning_rate = 1e-2
for it in range(10000):
    # 随机挑一个样本 id
    j = np.random.randint(1, len(X))

    # 选出对应的样本和标签
    example = X[j:j+1]
    label = Y[j]

    # 对这个样本做一次前向传播
    # 在这里写你的代码
    raise NotImplementedError()

    # 根据你的输出和标签计算损失
    # 在这里写你的代码
    raise NotImplementedError()
    
    # 反向传播
    # 在这里写你的代码
    raise NotImplementedError()
    
    # 梯度步
    # 在这里写你的代码
    raise NotImplementedError()

    # 每看过 250 个样本就画一次当前的决策边界
    if it % 250 == 0 : 
        plot_decision_boundary(ax, X,Y, net)
        fig.canvas.draw()
plot_decision_boundary(ax, X,Y, net)
fig.canvas.draw()

In [ ]:
%matplotlib inline
plt.plot(losses)

## 3. 使用 PyTorch 模块

在最后这部分，使用 `torch.nn.Module` 重新实现 `MyLinear` 和 `MyReLU`，让这些模块与 pytorch 兼容。


In [ ]:
import torch
import torch.nn as nn

# # y = xw + b
class MyLinear_mod(nn.Module):
    def __init__(self, n_input, n_output):
        super(MyLinear_mod, self).__init__()
        # 定义 self.A 和 self.b 作为权重和偏置
        # 用正态分布初始化它们
        # 用 nn.Parameters
        # 在这里写你的代码
        raise NotImplementedError()

    def forward(self, x):
        # 在这里写你的代码
        raise NotImplementedError()
        
class MyReLU_mod(nn.Module):
    def __init__(self):
        super(MyReLU_mod, self).__init__()
        
    def forward(self, x):
        # 在这里写你的代码
        raise NotImplementedError()

In [ ]:
# 用于绘制决策边界的网格现在应该由张量组成。
to_forward = torch.from_numpy(np.array(list(zip(xx.ravel(), yy.ravel())))).float()

用 `MyLinear_mod`、`MyReLU_mod` 和 [`nn.Sigmoid`](https://pytorch.org/docs/stable/nn.html#sigmoid) 定义你的网络


In [ ]:
h=50

# 用 nn.Sequential 定义网络
# 使用 MyLinear_mod、MyReLU_mod 和 nn.Sigmoid（来自 pytorch）
# 在这里写你的代码
raise NotImplementedError()

In [ ]:
from torch import optim
optimizer = optim.SGD(net.parameters(), lr=1e-2)

X_torch = torch.from_numpy(X).float()
Y_torch = torch.from_numpy(Y).float()

# 如果在 colab 上，你应该注释掉下面这一行
%matplotlib notebook
fig, ax = plt.subplots(1, 1, facecolor='#4B6EA9')
ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)

losses = []
criterion = nn.BCELoss()
for it in range(10000):
    # 随机挑一个样本 id
    j = np.random.randint(1, len(X))

    # 选出对应的样本和标签
    example = X_torch[j:j+1]
    label = Y_torch[j:j+1].unsqueeze(1)

    # 对这个样本做一次前向传播
    # 在这里写你的代码
    raise NotImplementedError()

    # 根据你的输出和标签计算损失
    # 在这里写你的代码
    raise NotImplementedError()

    # 梯度清零
    # 在这里写你的代码
    raise NotImplementedError()

    # 反向传播
    # 在这里写你的代码
    raise NotImplementedError()

    # 梯度步
    # 在这里写你的代码
    raise NotImplementedError()

    # 每看过 250 个样本就画一次当前的决策边界
    if it % 250 == 0 : 
        plot_decision_boundary(ax, X,Y, net)
        fig.canvas.draw()
plot_decision_boundary(ax, X,Y, net)
fig.canvas.draw()

In [ ]:
%matplotlib inline
plt.plot(losses)